# Checking metadata against a BattINFO schema

The six `battinfo_*` fields of `Metadata` are plain dicts in the shape of their BattINFO schema. Nothing checks
a value when you assign it. `Metadata.check()`, `read()` and `save()` check the fields against the schemas of
a BattINFO schema with `jsonschema`. This notebook uses only the bundled BattINFO schema, so it fetches nothing.

In [ ]:
import tempfile
import warnings
from pathlib import Path

import polars as pl

import bdf
from bdf.battinfo import BATTINFO_SCHEMA, BattinfoSchema

work = Path(tempfile.mkdtemp())
frame = pl.DataFrame({"Test Time / s": [0.0, 1.0], "Voltage / V": [3.6, 3.61], "Current / A": [0.0, 0.5]})

## Dict access and `check()`

Write an entity the way BattINFO writes it. `check()` returns one line for each refused value, and never raises.

In [ ]:
meta = bdf.Metadata()
meta.battinfo_test = {"test": {"name": "cell-1 formation", "instrument_name": "BaSyTec CTS"}}
print("valid:", meta.check())

meta.battinfo_test["test"]["instrument_name"] = 42  # a string is expected
for line in meta.check():
    print("refused:", line)

## The read and save errors

A save that the check refuses raises `BDFMetadataError` and writes nothing. A read of a sidecar does the same.

In [ ]:
artifact = work / "cell01.bdf.csv"
try:
    bdf.save(frame, artifact, metadata=meta)
except bdf.BDFMetadataError as exc:
    print(exc)
print("files written:", sorted(p.name for p in work.iterdir()))

## `strict=False` and `validate=False`

With `strict=False` the same text is a `BDFMetadataWarning`, and both files are written. With `validate=False` nothing is checked: no warning, and both files are written.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    bdf.save(frame, artifact, metadata=meta, strict=False)
print([w.category.__name__ for w in caught])
print("files written:", sorted(p.name for p in work.iterdir()))

## A named BattINFO schema version

`BattinfoSchema.from_version()` loads a BattINFO schema: from the bundle, the cache, or upstream.
`check(schema)` checks against exactly that schema, and `BATTINFO_SCHEMA`, the bundled BattINFO schema, does not change.

In [ ]:
schema = BattinfoSchema.from_version("0.8.0rc2")
print("named release:  ", schema.version)
print("refused by it:  ", len(meta.check(schema)), "value")
print("bundled release:", BATTINFO_SCHEMA.version)

## A save that states its BattINFO schema version

`meta.bdf.battinfo_version` names the BattINFO schema the fields follow. A read and a save check
against it, and the sidecar's `bdf.battinfo_version` states the version that bdf checked against.

In [ ]:
meta.battinfo_test["test"]["instrument_name"] = "BaSyTec CTS"
meta.bdf.battinfo_version = "0.8.0rc2"
stated = work / "cell02.bdf.csv"
bdf.save(frame, stated, metadata=meta)
_, restored = bdf.read(stated)
print("read back checked against:", restored.bdf.battinfo_version)